In [4]:
using DifferentialEquations

function rossler!(du, u, (a, b, c), t)
    x, y, z = u
    du .= (-y - z, x + a*y, b + z*(x - c))
end

function compute_section(; MaxPts = 49)
    problem = ODEProblem(
        rossler!,
        [1.0, 0.0, 0.0],
        (0.0, 1000.0),
        (0.2, 0.2, 5.7)
    )

    points = Matrix{Float64}(undef, 3, MaxPts)
    point_count = 0

    callback = ContinuousCallback(
        (u, t, integrator) -> u[1] - u[2],
        nothing,
        integrator -> begin
            point_count += 1
            points[:, point_count] .= integrator.u
            point_count == MaxPts && terminate!(integrator)
        end;
        save_positions = (false, false)
    )

    solution = solve(
        problem,
        Tsit5();
        callback,
        abstol = 1e-9,
        reltol = 1e-9
    )

    return solution, points, point_count
end

solution, points, point_count = compute_section(MaxPts = 49)

@assert point_count == 49
println("Saved $point_count section points.")
println("Final point: ", points[:, end])
println("Solver return code: ", solution.retcode)

Saved 49 section points.
Final point: [3.1995317227897964, 3.1995317227894824, 0.11204605727975943]
Solver return code: Terminated


Tsit5 accepted step
    → dense Tsit5 interpolation
    → detect a +/− bracket for x(t)−y(t)
    → bracketed scalar root search on the interpolated condition
    → move integrator to the selected floating-point side of the root
    → run your negative-crossing action